# 08.2 — SHAP Protetor: Modelo LightGBM + Dependence Plots

**Purpose:** SHAP model with proper train/test split, dependence plots,
temporal stability of protective profiles, waterfall for low-risk cases.

Requires artifacts from 08.1: `out/artifacts_menor_risco/df_doc.pkl`
Outputs → `out/08.2_shap_protetor/`

In [ ]:
from __future__ import annotations
import json
import pickle
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats as scipy_stats

warnings.filterwarnings("ignore")

ROOT    = Path(__file__).resolve().parent.parent
ART_DIR = ROOT / "out" / "artifacts_menor_risco"
OUT_DIR = ROOT / "out" / "08.2_shap_protetor"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Idioma / Language ─────────────────────────────────────────────────────────
LANG = "pt"  # "pt" | "en"  ← única linha a alterar para mudar idioma

_T = {
    "pt": {
        "shap_xlabel":      "SHAP médio (negativo = protetor, positivo = risco)",
        "shap_title":       "Features Protetoras e de Risco — LightGBM (test set)\n(negativo = reduz risco de retorno)",
        "dep_ylabel":       "SHAP value",
        "dep_suptitle":     "SHAP Dependence — Top 3 Features Protetoras (test set)",
        "wf_suptitle":      "SHAP Waterfall — 3 Casos de Menor Risco (test set)\n(valores negativos = contribuição protetora)",
        "wf_xlabel":        "SHAP value",
        "temp_xlabel":      "Ano",
        "temp_ylabel":      "Taxa de retorno 30d (%)",
        "temp_title":       "Estabilidade Temporal dos Perfis Protetores\n(Documentadas — por ano)",
        "temp_base_lbl":    "Base (all documented)",
    },
    "en": {
        "shap_xlabel":      "Mean SHAP (negative = protective, positive = risk)",
        "shap_title":       "Protective and Risk Features — LightGBM (test set)\n(negative = reduces return risk)",
        "dep_ylabel":       "SHAP value",
        "dep_suptitle":     "SHAP Dependence — Top 3 Protective Features (test set)",
        "wf_suptitle":      "SHAP Waterfall — 3 Lowest-Risk Cases (test set)\n(negative values = protective contribution)",
        "wf_xlabel":        "SHAP value",
        "temp_xlabel":      "Year",
        "temp_ylabel":      "30-day return rate (%)",
        "temp_title":       "Temporal Stability of Protective Profiles\n(Documented patients — by year)",
        "temp_base_lbl":    "Base (all documented)",
    },
}
T = _T[LANG]

## Load Artifacts from 08.1

In [ ]:
art_path = ART_DIR / "df_doc.pkl"
if not art_path.exists():
    raise FileNotFoundError(
        f"Artifact not found: {art_path}\n"
        "Run 08.1_perfil_documentados.py first."
    )

with open(art_path, "rb") as f:
    _art = pickle.load(f)

df              = _art["df"]
df_doc          = _art["df_doc"]
ref             = _art["ref"]
FEATURES        = _art["FEATURES"]
top3_protetores = _art.get("top3_protetores", [])
cond_df         = _art.get("cond_df", pd.DataFrame())

print(f"Loaded: df={len(df):,}, df_doc={len(df_doc):,}")
print(f"Ref (retorno 30d média): {ref:.1f}%")
print(f"FEATURES ({len(FEATURES)}): {FEATURES}")
print(f"Top 3 protetores: {top3_protetores}")

## Section 1 — LightGBM with Temporal Train/Test Split

In [ ]:
try:
    import shap
    from lightgbm import LGBMClassifier
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    from sklearn.impute import SimpleImputer
    from sklearn.experimental import enable_iterative_imputer  # noqa: F401
    from sklearn.impute import IterativeImputer
    from sklearn.metrics import roc_auc_score, average_precision_score

    # Temporal split: train ≤ 2021, test ≥ 2022
    if "ano" not in df_doc.columns:
        df_doc = df_doc.copy()
        if "dt_atendimento" in df_doc.columns:
            df_doc["ano"] = pd.to_datetime(df_doc["dt_atendimento"], errors="coerce").dt.year
        else:
            df_doc["ano"] = 2021  # fallback: no split possible

    train_mask = df_doc["ano"] <= 2021
    test_mask  = df_doc["ano"] >= 2022

    df_train = df_doc[train_mask].copy()
    df_test  = df_doc[test_mask].copy()

    # If test set is too small, fall back to 80/20 index split
    if len(df_test) < 50:
        print("Test set too small for temporal split — using last 20% as test")
        n = len(df_doc)
        cutoff = int(n * 0.8)
        df_train = df_doc.iloc[:cutoff].copy()
        df_test  = df_doc.iloc[cutoff:].copy()

    print(f"Train: {len(df_train):,} | Test: {len(df_test):,}")
    print(f"Train retorno: {df_train['y'].mean()*100:.1f}%  "
          f"| Test retorno: {df_test['y'].mean()*100:.1f}%")

    X_train = df_train[FEATURES]
    y_train = df_train["y"]
    X_test  = df_test[FEATURES]
    y_test  = df_test["y"]

    num_cols = [c for c in FEATURES if pd.api.types.is_numeric_dtype(X_train[c])]
    cat_cols = [c for c in FEATURES if c not in num_cols]

    def build_pre(X):
        """Build ColumnTransformer with MICE for numerics."""
        _num = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
        _cat = [c for c in X.columns if c not in _num]
        return ColumnTransformer([
            ("num", Pipeline([
                ("imp", IterativeImputer(max_iter=10, random_state=42, min_value=0)),
                ("sc",  StandardScaler()),
            ]), _num),
            ("cat", Pipeline([
                ("imp", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]), _cat),
        ], remainder="drop")

    pre = build_pre(X_train)
    X_train_t = pre.fit_transform(X_train)
    X_test_t  = pre.transform(X_test)

    feat_names = [n.split("__", 1)[-1] for n in pre.get_feature_names_out()]

    clf = LGBMClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        n_jobs=-1, random_state=42, verbose=-1,
    )
    clf.fit(X_train_t, y_train)

    # Evaluate on test set
    y_pred_proba = clf.predict_proba(X_test_t)[:, 1]
    auroc  = roc_auc_score(y_test, y_pred_proba)
    auprc  = average_precision_score(y_test, y_pred_proba)
    print(f"\nTest AUROC: {auroc:.4f}  |  Test AUPRC: {auprc:.4f}")

    metricas = pd.DataFrame([{
        "metrica"   : "AUROC",  "valor": round(auroc, 4),
        "split_tipo": "temporal",
        "n_train"   : len(df_train), "n_test": len(df_test),
    }, {
        "metrica"   : "AUPRC",  "valor": round(auprc, 4),
        "split_tipo": "temporal",
        "n_train"   : len(df_train), "n_test": len(df_test),
    }])
    metricas.to_csv(OUT_DIR / "metricas_modelo_doc.csv", index=False)
    print(f"Saved: {OUT_DIR / 'metricas_modelo_doc.csv'}")

    _shap_available = True

except ImportError as e:
    print(f"SHAP/LightGBM não disponível: {e}")
    _shap_available = False
    auroc = auprc = np.nan

## Section 2 — SHAP Global (test set only)

In [ ]:
if _shap_available:
    # Use ONLY test set for SHAP explanations
    rng = np.random.default_rng(42)
    n_shap = min(800, len(X_test_t))
    idx_s  = rng.integers(0, len(X_test_t), n_shap)
    X_s    = np.asarray(X_test_t[idx_s])

    explainer = shap.TreeExplainer(clf)
    sv = explainer.shap_values(X_s)
    if isinstance(sv, list): sv = sv[1]
    if getattr(sv, "ndim", 0) == 3: sv = sv[:, :, 1]
    sv = np.asarray(sv)

    # Signed mean (direction) and absolute mean (magnitude)
    shap_mean = sv.mean(axis=0)
    shap_abs  = np.abs(sv).mean(axis=0)

    shap_df = pd.DataFrame({
        "feature"    : feat_names,
        "shap_medio" : shap_mean,
        "shap_abs"   : shap_abs,
        "direcao"    : ["PROTETOR" if v < -0.005 else
                        "RISCO"    if v >  0.005 else "NEUTRO"
                        for v in shap_mean],
    }).sort_values("shap_medio").reset_index(drop=True)

    protetores_shap = shap_df[shap_df["direcao"] == "PROTETOR"].head(15)
    risco_shap      = shap_df[shap_df["direcao"] == "RISCO"].head(10)

    print("\n" + "=" * 60)
    print("TOP FEATURES PROTETORAS (SHAP médio negativo) — test set")
    print("=" * 60)
    print(protetores_shap[["feature", "shap_medio", "shap_abs"]].to_string(index=False))
    print("\nTOP FEATURES DE RISCO (SHAP médio positivo) — test set")
    print(risco_shap[["feature", "shap_medio", "shap_abs"]].to_string(index=False))

    # Bidirectional bar chart: top 10 protective + top 10 risk
    top_prot = shap_df[shap_df["direcao"] == "PROTETOR"].head(10)
    top_risk = shap_df[shap_df["direcao"] == "RISCO"].head(10)
    top_n = pd.concat([top_prot, top_risk]).sort_values("shap_medio")

    fig, ax = plt.subplots(figsize=(10, 8))
    cores_s = ["#2ca02c" if v < 0 else "#d62728" for v in top_n["shap_medio"]]
    ax.barh(top_n["feature"], top_n["shap_medio"], color=cores_s, alpha=0.85)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel(T["shap_xlabel"])
    ax.set_title(T["shap_title"])
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "03_shap_protetor_risco.png", dpi=150)
    plt.close()
    print(f"Saved: {OUT_DIR / '03_shap_protetor_risco.png'}")

    shap_df.to_csv(OUT_DIR / "shap_protetor_risco.csv", index=False)
    print(f"Saved: {OUT_DIR / 'shap_protetor_risco.csv'}")

## Section 3 — SHAP Dependence Plots for Top 3 Protective Features

In [ ]:
if _shap_available:
    # Top 3 features with most negative SHAP mean (most protective)
    top3_feat = (
        shap_df[shap_df["direcao"] == "PROTETOR"]
        .head(3)["feature"]
        .tolist()
    )
    print(f"\nTop 3 protective features for dependence plots: {top3_feat}")

    if top3_feat:
        fig, axes = plt.subplots(1, len(top3_feat), figsize=(6 * len(top3_feat), 5))
        if len(top3_feat) == 1:
            axes = [axes]

        for ax, feat in zip(axes, top3_feat):
            if feat not in feat_names:
                ax.set_title(f"{feat}\n(not in feat_names)")
                continue
            feat_idx = feat_names.index(feat)

            # Find most interacting feature (highest |correlation| of SHAP values
            # with raw feature values in X_s)
            shap_col = sv[:, feat_idx]
            best_interact = None
            best_corr = -1.0
            for j, fn in enumerate(feat_names):
                if j == feat_idx:
                    continue
                try:
                    r, _ = scipy_stats.spearmanr(X_s[:, j], shap_col, nan_policy="omit")
                    if abs(r) > best_corr:
                        best_corr = abs(r)
                        best_interact = j
                except Exception:
                    pass

            feat_vals = X_s[:, feat_idx]
            color_vals = X_s[:, best_interact] if best_interact is not None else np.zeros(len(X_s))

            sc = ax.scatter(feat_vals, shap_col, c=color_vals, cmap="coolwarm",
                            alpha=0.5, s=15, linewidths=0)
            ax.axhline(0, color="black", lw=0.8, ls="--")
            ax.set_xlabel(feat, fontsize=9)
            ax.set_ylabel(T["dep_ylabel"], fontsize=9)
            interact_name = feat_names[best_interact] if best_interact is not None else "N/A"
            ax.set_title(f"{feat}\n(cor: {interact_name[:20]})", fontsize=9)
            plt.colorbar(sc, ax=ax, label=interact_name[:15])

        fig.suptitle(T["dep_suptitle"], fontsize=11)
        plt.tight_layout()
        plt.savefig(OUT_DIR / "04_shap_dependence_protetoras.png", dpi=150)
        plt.close()
        print(f"Saved: {OUT_DIR / '04_shap_dependence_protetoras.png'}")
    else:
        print("No protective features found for dependence plots.")

## Section 4 — SHAP Waterfall for 3 Lowest-Risk Cases

In [ ]:
if _shap_available:
    # Find 3 cases in test set with lowest predicted probability
    lowest_idx = np.argsort(y_pred_proba)[:3]
    print(f"\nLowest-risk cases (test set indices): {lowest_idx}")
    print(f"Predicted probabilities: {y_pred_proba[lowest_idx]}")

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    for ax, case_idx in zip(axes, lowest_idx):
        x_case = X_s[case_idx] if case_idx < len(X_s) else X_test_t[case_idx]
        sv_case = sv[case_idx] if case_idx < len(sv) else explainer.shap_values(
            X_test_t[case_idx:case_idx+1]
        )
        if isinstance(sv_case, list): sv_case = sv_case[1]
        if getattr(sv_case, "ndim", 0) == 2:
            if sv_case.shape[-1] == 2:
                sv_case = sv_case[:, 1]
            sv_case = sv_case.flatten()

        # Manual waterfall: top N features by |SHAP|
        n_show = 10
        abs_sv = np.abs(sv_case)
        top_idx = np.argsort(abs_sv)[::-1][:n_show]
        top_sv  = sv_case[top_idx]
        top_fn  = [feat_names[i] for i in top_idx]

        base = float(explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value)
        cumsum = base + np.cumsum(np.append(0, top_sv))

        colors_wf = ["#2ca02c" if v < 0 else "#d62728" for v in top_sv]
        ax.barh(top_fn[::-1], top_sv[::-1], color=colors_wf[::-1], alpha=0.85)
        ax.axvline(0, color="black", lw=0.7)
        ax.set_title(f"Case {case_idx}\np={y_pred_proba[case_idx]:.3f}", fontsize=9)
        ax.set_xlabel(T["wf_xlabel"], fontsize=8)
        ax.tick_params(axis="y", labelsize=7)
        ax.grid(axis="x", alpha=0.3)

    fig.suptitle(T["wf_suptitle"], fontsize=10)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "05_shap_waterfall_baixo_risco.png", dpi=150)
    plt.close()
    print(f"Saved: {OUT_DIR / '05_shap_waterfall_baixo_risco.png'}")

## Section 5 — Temporal Stability of Protective Profiles

In [ ]:
if top3_protetores and "ano" in df_doc.columns:
    print(f"\nTemporal stability for: {top3_protetores}")

    stab_rows = []

    # Map condition names back to boolean masks on df_doc
    def _cond_mask(nome, d):
        ig_ = pd.to_numeric(d.get("ig_obs", d.get("semanas_gestacao")), errors="coerce")
        if nome.startswith("tipo="):
            return d.get("tipo_gestacao", pd.Series([""] * len(d), index=d.index)) == nome.split("=", 1)[1]
        if nome.startswith("perfil="):
            return d.get("perfil_historico", pd.Series([""] * len(d), index=d.index)).fillna("") == nome.split("=", 1)[1]
        if nome == "IG 37-40s (termo)":
            return ig_.between(37, 40, inclusive="both")
        if nome == "Sem red_flag (LLM)":
            return pd.to_numeric(d.get("red_flag", pd.Series([-1]*len(d), index=d.index)), errors="coerce").fillna(-1) == 0
        if nome.startswith("status="):
            return d.get("clinical_status", pd.Series([""] * len(d), index=d.index)).fillna("") == nome.split("=", 1)[1]
        if nome.startswith("severity="):
            return d.get("severity", pd.Series([""] * len(d), index=d.index)).fillna("") == nome.split("=", 1)[1]
        if nome == "Manchester verde/amarelo":
            return pd.to_numeric(d.get("cd_cor_manchester", pd.Series([99]*len(d), index=d.index)), errors="coerce").fillna(99) <= 2
        if nome == "Pré-natal ≥ 6 consultas":
            return pd.to_numeric(d.get("nr_consultas_pre_natal", pd.Series([0]*len(d), index=d.index)), errors="coerce") >= 6
        if nome == "Primigesta + IG 37-40s":
            _pg = d.get("primigesta", pd.Series([0]*len(d), index=d.index)).fillna(0)
            return (_pg == 1) & ig_.reindex(d.index, fill_value=np.nan).between(37, 40)
        return pd.Series([False] * len(d), index=d.index)

    years = sorted(df_doc["ano"].dropna().unique().astype(int))
    for year in years:
        sub_yr = df_doc[df_doc["ano"] == year]
        row = {"ano": year, "n_total_yr": len(sub_yr),
               "ret_base_pct": round(sub_yr["y"].mean() * 100, 2)}
        for nome in top3_protetores:
            try:
                mask = _cond_mask(nome, sub_yr)
                sub_cond = sub_yr[mask]
                row[f"ret_{nome[:30]}_pct"] = round(sub_cond["y"].mean() * 100, 2) if len(sub_cond) >= 10 else np.nan
                row[f"n_{nome[:30]}"]       = int(len(sub_cond))
            except Exception as e:
                row[f"ret_{nome[:30]}_pct"] = np.nan
                row[f"n_{nome[:30]}"]       = 0
        stab_rows.append(row)

    stab_df = pd.DataFrame(stab_rows)
    stab_df.to_csv(OUT_DIR / "estabilidade_temporal.csv", index=False)
    print(stab_df.to_string(index=False))

    # Line plot
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(stab_df["ano"], stab_df["ret_base_pct"],
            marker="o", ls="--", color="gray", label=T["temp_base_lbl"], lw=1.5)
    colors_stab = ["#2ca02c", "#1f77b4", "#ff7f0e"]
    for nome, col in zip(top3_protetores, colors_stab):
        col_name = f"ret_{nome[:30]}_pct"
        if col_name in stab_df.columns:
            ax.plot(stab_df["ano"], stab_df[col_name],
                    marker="s", label=nome[:40], color=col, lw=1.8)
    ax.set_xlabel(T["temp_xlabel"])
    ax.set_ylabel(T["temp_ylabel"])
    ax.set_title(T["temp_title"])
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "06_estabilidade_temporal_protetores.png", dpi=150)
    plt.close()
    print(f"Saved: {OUT_DIR / '06_estabilidade_temporal_protetores.png'}")
else:
    if not top3_protetores:
        print("top3_protetores vazio — temporal stability skipped.")
    if "ano" not in df_doc.columns:
        print("Coluna 'ano' ausente — temporal stability skipped.")

## Section 6 — Final Summary

In [ ]:
top_prot_shap = (
    shap_df[shap_df["direcao"] == "PROTETOR"]["feature"].head(3).tolist()
    if _shap_available and "shap_df" in dir() else []
)
top_risk_shap = (
    shap_df[shap_df["direcao"] == "RISCO"]["feature"].head(3).tolist()
    if _shap_available and "shap_df" in dir() else []
)

resumo = {
    "notebook"              : "08.2_shap_protetor",
    "data"                  : datetime.now().isoformat(),
    "n_documentadas"        : int(len(df_doc)),
    "taxa_retorno_30d_pct"  : round(ref, 2),
    "top3_protetores_08.1"  : top3_protetores,
    "top3_shap_protetores"  : top_prot_shap,
    "top3_shap_risco"       : top_risk_shap,
    "auroc_modelo_doc"      : round(float(auroc), 4) if not np.isnan(auroc) else None,
    "auprc_modelo_doc"      : round(float(auprc), 4) if not np.isnan(auprc) else None,
    "split_tipo"            : "temporal (train≤2021, test≥2022)",
    "n_train"               : int(len(df_train)) if "_shap_available" and _shap_available else None,
    "n_test"                : int(len(df_test))  if "_shap_available" and _shap_available else None,
}

(OUT_DIR / "resumo_menor_risco.json").write_text(
    json.dumps(resumo, ensure_ascii=False, indent=2), encoding="utf-8"
)

resumo_txt = (
    f"Análise: 08.2 SHAP Protetor\n"
    f"Data: {resumo['data']}\n"
    f"N documentadas: {resumo['n_documentadas']:,}\n"
    f"Taxa retorno média 30d: {resumo['taxa_retorno_30d_pct']}%\n"
    f"AUROC modelo: {resumo['auroc_modelo_doc']}\n"
    f"AUPRC modelo: {resumo['auprc_modelo_doc']}\n"
    f"Top protetores (08.1): {', '.join(top3_protetores)}\n"
    f"Top SHAP protetores: {', '.join(top_prot_shap)}\n"
    f"Top SHAP risco: {', '.join(top_risk_shap)}\n"
)
(OUT_DIR / "resumo_menor_risco.txt").write_text(resumo_txt, encoding="utf-8")

print()
print("╔══ 08 Perfil de Menor Risco — Resumo ══╗")
print(f"  N documentadas          : {resumo['n_documentadas']:,}")
print(f"  Taxa média retorno 30d  : {resumo['taxa_retorno_30d_pct']}%")
print(f"  Top condições protetoras: {top3_protetores}")
print(f"  Features SHAP protetoras: {top_prot_shap}")
print(f"  AUROC modelo documentadas: {resumo['auroc_modelo_doc']}")
print("╚═══════════════════════════════════════╝")

print(f"\nSaída: {OUT_DIR}")
print(f"  metricas_modelo_doc.csv")
print(f"  shap_protetor_risco.csv")
print(f"  estabilidade_temporal.csv")
print(f"  resumo_menor_risco.json / .txt")
print(f"  plots: 03_shap_protetor_risco.png, 04_shap_dependence_protetoras.png,")
print(f"         05_shap_waterfall_baixo_risco.png, 06_estabilidade_temporal_protetores.png")
print(f"\nConcluído — {datetime.now().isoformat()}")